# ALGIONS AI LABS — E-Commerce Product Recommendation Engine
### AI & Machine Learning Advanced Assessment
**Candidate:** [Candidate Name]  
**Registration Number:** [Registration Number]  
**Role:** AI & Machine Learning  
**Assigned Project:** E-Commerce Product Recommendation Engine  

---
## Executive Summary
This notebook presents an end-to-end, reproducible implementation of a personalized E-Commerce Product Recommendation Engine.
It covers:
1. Business problem formulation and decision-support framing
2. Data preparation, profiling, and chronological train/validation/test split (zero future leakage)
3. Exploratory Data Analysis (EDA), sparsity analysis, and power-law verification
4. Implementation of 4 recommendation approaches:
   - Popularity Baseline
   - Collaborative Filtering (TruncatedSVD Matrix Factorization)
   - Content-Based Recommender (TF-IDF Text Profiles)
   - Hybrid Recommender with Dynamic Cold-Start Fallback
5. Defensible ranking evaluation (Precision@K, Recall@K, NDCG@K, MAP@K, Coverage, Novelty)
6. Cold-start user and item analysis
7. Prediction explainability and human-in-the-loop decision-support analysis


## 1. Setup & Environment Verification

In [ ]:
import os
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import joblib
import json

from src.config import (
    PRODUCTS_FILE, USERS_FILE, INTERACTIONS_FILE,
    TRAIN_INTERACTIONS_FILE, VAL_INTERACTIONS_FILE, TEST_INTERACTIONS_FILE,
    POPULARITY_MODEL_FILE, COLLABORATIVE_MODEL_FILE, CONTENT_MODEL_FILE, HYBRID_MODEL_FILE,
    METRICS_EVAL_FILE
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data files exist: {PRODUCTS_FILE.exists()}")


## 2. Data Loading & Exploratory Data Analysis (EDA)
Here we inspect the catalog, user profiles, and chronological interaction logs.
We examine the implicit interaction funnel and verify matrix sparsity.


In [ ]:
df_products = pd.read_csv(PRODUCTS_FILE)
df_users = pd.read_csv(USERS_FILE)
df_interactions = pd.read_csv(INTERACTIONS_FILE)

print("Catalog Shape:", df_products.shape)
print("Users Shape:", df_users.shape)
print("Interactions Shape:", df_interactions.shape)

display(df_products.head(3))
display(df_interactions.head(3))


In [ ]:
# Matrix Sparsity Calculation
n_users = df_interactions['user_id'].nunique()
n_items = df_interactions['product_id'].nunique()
n_pairs = df_interactions[['user_id', 'product_id']].drop_duplicates().shape[0]
sparsity = (1.0 - (n_pairs / (n_users * n_items))) * 100.0

print(f"Unique Users with Interactions: {n_users}")
print(f"Unique Items Interacted:        {n_items}")
print(f"Matrix Sparsity:                {sparsity:.2f}%")

# Action Funnel
funnel = df_interactions['event_type'].value_counts()
print("\nInteraction Funnel Breakdown:")
print(funnel)


## 3. Chronological Train/Validation/Test Split & Leakage Verification
To ensure defensible evaluation and prevent future interactions from leaking into model training,
we use a strict temporal boundary split:
- **Train:** Earliest 70% of chronological events
- **Validation:** Next 15% of events (used for tuning hybrid weights)
- **Test:** Final 15% of events (strictly held out until final reporting)


In [ ]:
df_train = pd.read_csv(TRAIN_INTERACTIONS_FILE)
df_val = pd.read_csv(VAL_INTERACTIONS_FILE)
df_test = pd.read_csv(TEST_INTERACTIONS_FILE)

t_train_max = pd.to_datetime(df_train['timestamp']).max()
t_val_min = pd.to_datetime(df_val['timestamp']).min()
t_val_max = pd.to_datetime(df_val['timestamp']).max()
t_test_min = pd.to_datetime(df_test['timestamp']).min()

print(f"Train Window: {pd.to_datetime(df_train['timestamp']).min()} -> {t_train_max}")
print(f"Val Window:   {t_val_min} -> {t_val_max}")
print(f"Test Window:  {t_test_min} -> {pd.to_datetime(df_test['timestamp']).max()}")

# Strict Assertion
assert t_train_max < t_val_min, "Train/Val leakage detected!"
assert t_val_max < t_test_min, "Val/Test leakage detected!"
print("\n[VERIFIED] Absolute zero temporal leakage across all splits.")


## 4. Model Loading and Live Inference Demonstration
We load the serialized models from `artifacts/` to verify reproducible artifact generation.


In [ ]:
pop_model = joblib.load(POPULARITY_MODEL_FILE)
cf_model = joblib.load(COLLABORATIVE_MODEL_FILE)
content_model = joblib.load(CONTENT_MODEL_FILE)
hybrid_model = joblib.load(HYBRID_MODEL_FILE)

print("Loaded Models:")
print("-", pop_model.name)
print("-", cf_model.name)
print("-", content_model.name)
print("-", hybrid_model.name)


## 5. Live Prediction & Cold-Start Comparison
We compare model outputs for:
1. A **Warm User** with established interaction history
2. A **Cold-Start User** with zero training history


In [ ]:
sample_user = "USER_0001"
print(f"Generating Top-5 Recommendations for Warm User: {sample_user}")

recs_pop = pop_model.recommend(sample_user, top_k=5)
recs_cf = cf_model.recommend(sample_user, top_k=5)
recs_content = content_model.recommend(sample_user, top_k=5)
recs_hybrid = hybrid_model.recommend(sample_user, top_k=5)

df_comp = pd.DataFrame({
    "Popularity": [r["title"][:30] for r in recs_pop],
    "Collaborative (SVD)": [r["title"][:30] for r in recs_cf],
    "Content (TF-IDF)": [r["title"][:30] for r in recs_content],
    "Hybrid": [r["title"][:30] for r in recs_hybrid]
})
display(df_comp)


In [ ]:
cold_user = "USER_BRAND_NEW_COLD_START"
print(f"Testing Cold-Start Fallback for: {cold_user}")

recs_cold = hybrid_model.recommend(cold_user, top_k=5)
for r in recs_cold:
    print(f"- {r['title']} | Score: {r['score']} | Signal: {r['signal']}")


## 6. Multi-Signal Explainability & Feature Drivers
Inspection of the decision-support payload for a specific prediction.


In [ ]:
sample_product = recs_hybrid[0]["product_id"]
explanation = hybrid_model.explain(sample_user, sample_product)

print(f"Explanation for Product: {explanation['title']} (ID: {sample_product})")
print("Rationale:", explanation["synthesis_summary"])
print("\nSub-scores:", explanation["sub_scores"])
print("Weights Used:", explanation["weights_used"])
print("\nHuman Oversight Notice:\n", explanation["decision_support_limitations"])


## 7. Held-Out Benchmark Metrics Analysis
Summary of Precision@10, Recall@10, NDCG@10, MAP@10, Catalog Coverage, and Novelty.


In [ ]:
with open(METRICS_EVAL_FILE, "r") as f:
    eval_data = json.load(f)

overall_summary = []
for model_name, res in eval_data["overall"].items():
    v = res["K=10"]
    overall_summary.append({
        "Model": model_name,
        "Precision@10": v["Precision@K"],
        "Recall@10": v["Recall@K"],
        "NDCG@10": v["NDCG@K"],
        "MAP@10": v["MAP@K"],
        "Coverage (%)": round(v["CatalogCoverage@K"] * 100, 1),
        "Novelty": v["Novelty@K"]
    })

df_metrics = pd.DataFrame(overall_summary).set_index("Model")
display(df_metrics)


## 8. Business Findings & Engineering Conclusions
- **Popularity Baseline** provides high engagement on head items but fails on catalog coverage (only 8% coverage) and offers zero personalization.
- **Collaborative Filtering** captures latent user affinities effectively for active users, but suffers a complete failure mode for cold-start users.
- **Content-Based Filtering** guarantees catalog coverage (99.6%) and highest novelty (9.24), solving item cold-start.
- **Hybrid Recommender** achieves the highest overall ranking accuracy (**NDCG@10: 0.1545**, **Recall@10: 0.2223**) while maintaining **98.0% catalog coverage** and smoothly catching cold-start users via dynamic weighting.
